In [1]:
%pip install sentence-transformers scikit-learn pandas joblib

   ---------------------------------------- 0.0/780.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/780.3 kB ? eta -:--:--
   ---------------------------------------- 780.3/780.3 kB 4.0 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ==================================================
# Stage 04 - Pretrained Language Model
# Step 4.1 - Setup & Data Loading
# ==================================================

from pathlib import Path
import pandas as pd
import torch
import sentence_transformers

from sentence_transformers import SentenceTransformer

# Locate project automatically
CURRENT_DIR = Path.cwd().resolve()

PROJECT_DIR = next(
    (
        p for p in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (p / "data" / "processed" / "train.csv").exists()
    ),
    None
)

assert PROJECT_DIR is not None, "Project folder not found!"

DATA_DIR = PROJECT_DIR / "data" / "processed"

# Load datasets
train_df = pd.read_csv(DATA_DIR / "train.csv")
dev_df = pd.read_csv(DATA_DIR / "dev.csv")

# Prepare inputs and labels
X_train = train_df["text"]
y_train = train_df["label"]

X_dev = dev_df["text"]
y_dev = dev_df["label"]

# Configuration
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 55)
print("STAGE 04 - PRETRAINED LANGUAGE MODEL")
print("=" * 55)

print("Training Examples:", len(X_train))
print("Development Examples:", len(X_dev))
print("Target Classes:", y_train.nunique())
print("Pretrained Model:", MODEL_NAME)
print("Device:", DEVICE)
print("Sentence Transformers:", sentence_transformers.__version__)

assert len(X_train) == 3558
assert len(X_dev) == 762

print("\nSetup Verification: PASSED!")

W1118 01:41:52.563000 41072 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


STAGE 04 - PRETRAINED LANGUAGE MODEL
Training Examples: 3558
Development Examples: 762
Target Classes: 5
Pretrained Model: sentence-transformers/all-MiniLM-L6-v2
Device: cpu
Sentence Transformers: 6.3.0

Setup Verification: PASSED!


In [3]:
# ==================================================
# Step 4.2 - Load Pretrained MiniLM Model
# ==================================================

model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE
)

print("=" * 55)
print("PRETRAINED MODEL INFORMATION")
print("=" * 55)

print("Model:", MODEL_NAME)
print("Device:", model.device)

print(
    "Embedding Dimension:",
    model.get_sentence_embedding_dimension()
)

print(
    "Max Sequence Length:",
    model.max_seq_length
)

assert model.get_sentence_embedding_dimension() == 384

print("\nMiniLM Loading: PASSED!")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\faisa\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\faisa\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (7 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

PRETRAINED MODEL INFORMATION
Model: sentence-transformers/all-MiniLM-L6-v2
Device: cpu
Embedding Dimension: 384
Max Sequence Length: 256

MiniLM Loading: PASSED!


C:\Users\faisa\AppData\Local\Temp\ipykernel_41072\4259291420.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()
C:\Users\faisa\AppData\Local\Temp\ipykernel_41072\4259291420.py:27: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert model.get_sentence_embedding_dimension() == 384


In [4]:
# ==================================================
# Step 4.2 - Test Sentence Embeddings
# ==================================================

import numpy as np

# Select three descriptions
sample_texts = X_train.iloc[:3].tolist()

# Convert texts into embeddings
sample_embeddings = model.encode(
    sample_texts,
    batch_size=3,
    convert_to_numpy=True,
    show_progress_bar=False
)

print("=" * 55)
print("SENTENCE EMBEDDING TEST")
print("=" * 55)

print("Number of Texts:", len(sample_texts))
print("Embedding Shape:", sample_embeddings.shape)
print("Data Type:", sample_embeddings.dtype)

print("\nFirst 10 Embedding Values:")
print(np.round(sample_embeddings[0][:10], 4))

# Verification
assert sample_embeddings.shape == (3, 384)
assert np.isfinite(sample_embeddings).all()

print("\nEmbedding Generation: PASSED!")


SENTENCE EMBEDDING TEST
Number of Texts: 3
Embedding Shape: (3, 384)
Data Type: float32

First 10 Embedding Values:
[-0.0014  0.0266  0.1018 -0.0367  0.0806  0.0077  0.0203 -0.0134  0.0428
  0.0195]

Embedding Generation: PASSED!


In [5]:
# ==================================================
# Step 4.3 - Token Length Analysis
# ==================================================

import numpy as np
import pandas as pd

# Use Training and Development only
datasets = {
    "Training": X_train.tolist(),
    "Development": X_dev.tolist()
}

MAX_TOKENS = model.max_seq_length
BATCH_SIZE = 128

print("=" * 55)
print("TOKEN LENGTH ANALYSIS")
print("=" * 55)

print("Model Token Limit:", MAX_TOKENS)

token_statistics = []

for dataset_name, texts in datasets.items():

    lengths = []

    # Tokenize in batches without truncation
    for start in range(0, len(texts), BATCH_SIZE):

        batch = texts[start:start + BATCH_SIZE]

        encoded = model.tokenizer(
            batch,
            add_special_tokens=True,
            truncation=False,
            padding=False,
            return_length=True
        )

        lengths.extend(encoded["length"])

    lengths = np.array(lengths)

    # Count texts exceeding model limit
    long_texts = int((lengths > MAX_TOKENS).sum())

    percentage = long_texts / len(texts) * 100

    token_statistics.append({
        "Dataset": dataset_name,
        "Total": len(texts),
        "Min Tokens": int(lengths.min()),
        "Median Tokens": float(np.median(lengths)),
        "95th Percentile": float(np.percentile(lengths, 95)),
        "Max Tokens": int(lengths.max()),
        "Over Limit": long_texts,
        "Over Limit %": round(percentage, 2)
    })

# Display results
stats_df = pd.DataFrame(token_statistics)

print("\nDataset Token Statistics:")
display(stats_df)

print("\nAnalysis completed!")


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (376 > 256). Running this sequence through the model will result in indexing errors


TOKEN LENGTH ANALYSIS
Model Token Limit: 256

Dataset Token Statistics:


,Dataset,Total,Min Tokens,Median Tokens,95th Percentile,Max Tokens,Over Limit,Over Limit %
0,Training,3558,26,65.0,137.00,666,39,1.10
1,Development,762,26,66.0,139.95,454,9,1.18



Analysis completed!


In [6]:
# ============================================================
# Step 4.4 - Generate Sentence Embeddings
# ============================================================

import numpy as np

BATCH_SIZE = 32

print("=" * 55)
print("SENTENCE EMBEDDING GENERATION")
print("=" * 55)

print("Model:", MODEL_NAME)
print("Device:", DEVICE)
print("Batch Size:", BATCH_SIZE)

# 1. Generate Training Embeddings
print("\nGenerating Training Embeddings...")

X_train_embeddings = model.encode(
    X_train.tolist(),
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Training Embeddings Generated!")

# 2. Generate Development Embeddings
print("\nGenerating Development Embeddings...")

X_dev_embeddings = model.encode(
    X_dev.tolist(),
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Development Embeddings Generated!")

# 3. Display shapes
print("\n" + "=" * 55)
print("EMBEDDING GENERATION RESULTS")
print("=" * 55)

print("Training Shape:", X_train_embeddings.shape)
print("Development Shape:", X_dev_embeddings.shape)

print("Data Type:", X_train_embeddings.dtype)

print("\nEmbedding generation completed!")


SENTENCE EMBEDDING GENERATION
Model: sentence-transformers/all-MiniLM-L6-v2
Device: cpu
Batch Size: 32

Generating Training Embeddings...


Batches:   0%|          | 0/112 [00:00<?, ?it/s]

Training Embeddings Generated!

Generating Development Embeddings...


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Development Embeddings Generated!

EMBEDDING GENERATION RESULTS
Training Shape: (3558, 384)
Development Shape: (762, 384)
Data Type: float32

Embedding generation completed!


In [7]:
# ============================================================
# Step 4.4 - Embedding Verification
# ============================================================

print("=" * 55)
print("EMBEDDING VERIFICATION")
print("=" * 55)

# 1. Check dimensions
assert X_train_embeddings.shape == (3558, 384)
assert X_dev_embeddings.shape == (762, 384)

# 2. Check numerical validity
assert np.isfinite(X_train_embeddings).all()
assert np.isfinite(X_dev_embeddings).all()

# 3. Check vector normalization
train_norms = np.linalg.norm(
    X_train_embeddings,
    axis=1
)

dev_norms = np.linalg.norm(
    X_dev_embeddings,
    axis=1
)

assert np.allclose(train_norms, 1.0, atol=1e-4)
assert np.allclose(dev_norms, 1.0, atol=1e-4)

# 4. Display statistics
print("Training Shape:", X_train_embeddings.shape)
print("Development Shape:", X_dev_embeddings.shape)

print("\nTraining Mean Norm:", train_norms.mean())
print("Development Mean Norm:", dev_norms.mean())

print("\nTraining Memory (MB):",
      round(X_train_embeddings.nbytes / 1024**2, 2))

print("Development Memory (MB):",
      round(X_dev_embeddings.nbytes / 1024**2, 2))

print("\nFirst 10 Values:")
print(np.round(X_train_embeddings[0][:10], 4))

print("\nEmbedding Verification: PASSED!")


EMBEDDING VERIFICATION
Training Shape: (3558, 384)
Development Shape: (762, 384)

Training Mean Norm: 1.0
Development Mean Norm: 1.0

Training Memory (MB): 5.21
Development Memory (MB): 1.12

First 10 Values:
[-0.0014  0.0266  0.1018 -0.0367  0.0806  0.0077  0.0203 -0.0134  0.0428
  0.0195]

Embedding Verification: PASSED!


In [8]:
# ============================================================
# Step 4.5 - MiniLM + Logistic Regression Training
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# 1. Initialize classifier
minilm_baseline = LogisticRegression(
    C=1.0,
    max_iter=2000,
    random_state=42
)

# 2. Train classifier using MiniLM Embeddings
minilm_baseline.fit(
    X_train_embeddings,
    y_train
)

print("MiniLM classifier training completed!")

# 3. Predict Development Set
dev_predictions_minilm = minilm_baseline.predict(
    X_dev_embeddings
)

# 4. Calculate metrics
minilm_dev_accuracy = accuracy_score(
    y_dev,
    dev_predictions_minilm
)

minilm_dev_macro_f1 = f1_score(
    y_dev,
    dev_predictions_minilm,
    average="macro"
)

# 5. Display results
print("\n" + "=" * 55)
print("MINILM INITIAL DEVELOPMENT RESULTS")
print("=" * 55)

print(f"Accuracy: {minilm_dev_accuracy:.4f}")
print(f"Macro F1: {minilm_dev_macro_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_dev,
        dev_predictions_minilm,
        digits=4,
        zero_division=0
    )
)

# 6. Verification
assert len(dev_predictions_minilm) == len(y_dev)
assert len(minilm_baseline.classes_) == 5

print("\nMiniLM baseline verification: PASSED!")


MiniLM classifier training completed!

MINILM INITIAL DEVELOPMENT RESULTS
Accuracy: 0.9383
Macro F1: 0.9384

Classification Report:
                   precision    recall  f1-score   support

         Air Bags     0.9727    0.9068    0.9386       118
Electrical System     0.9103    0.9682    0.9384       283
      Fuel System     0.9444    0.9273    0.9358       110
   Service Brakes     0.9510    0.9252    0.9379       147
         Steering     0.9600    0.9231    0.9412       104

         accuracy                         0.9383       762
        macro avg     0.9477    0.9301    0.9384       762
     weighted avg     0.9395    0.9383    0.9383       762


MiniLM baseline verification: PASSED!


In [9]:
# ============================================================
# Step 4.6 - MiniLM Hyperparameter Tuning
# ============================================================

import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Hyperparameter configurations
C_VALUES = [0.1, 1.0, 3.0, 10.0]
CLASS_WEIGHTS = [None, "balanced"]

minilm_results = []
minilm_trained_models = {}

print("=" * 55)
print("MINILM HYPERPARAMETER TUNING")
print("=" * 55)

# Train and evaluate each configuration
for c in C_VALUES:
    for weight in CLASS_WEIGHTS:

        classifier = LogisticRegression(
            C=c,
            class_weight=weight,
            max_iter=2000,
            random_state=42
        )

        # Train on MiniLM training embeddings
        classifier.fit(
            X_train_embeddings,
            y_train
        )

        # Predict development embeddings
        predictions = classifier.predict(
            X_dev_embeddings
        )

        # Calculate metrics
        accuracy = accuracy_score(
            y_dev,
            predictions
        )

        macro_f1 = f1_score(
            y_dev,
            predictions,
            average="macro"
        )

        # Store results
        minilm_results.append({
            "C": c,
            "class_weight": str(weight),
            "accuracy": accuracy,
            "macro_f1": macro_f1
        })

        minilm_trained_models[(c, weight)] = classifier

        print(
            f"C={c:<5} | "
            f"Weight={str(weight):<8} | "
            f"Accuracy={accuracy:.4f} | "
            f"Macro F1={macro_f1:.4f}"
        )

# ============================================================
# Rank configurations
# ============================================================

minilm_tuning_results = pd.DataFrame(
    minilm_results
).sort_values(
    by=["macro_f1", "accuracy"],
    ascending=[False, False],
    kind="stable"
).reset_index(drop=True)

print("\n" + "=" * 55)
print("MINILM TUNING RESULTS - RANKED")
print("=" * 55)

display(minilm_tuning_results)

# ============================================================
# Select best classifier
# ============================================================

best_config_minilm = minilm_tuning_results.iloc[0]

best_c_minilm = float(best_config_minilm["C"])

best_weight_minilm = (
    None
    if best_config_minilm["class_weight"] == "None"
    else "balanced"
)

best_minilm_classifier = minilm_trained_models[
    (best_c_minilm, best_weight_minilm)
]

print("\n" + "=" * 55)
print("BEST MINILM CLASSIFIER")
print("=" * 55)

print("Best C:", best_c_minilm)
print("Best Class Weight:", best_weight_minilm)

print(
    "Best Development Accuracy:",
    round(best_config_minilm["accuracy"], 4)
)

print(
    "Best Development Macro F1:",
    round(best_config_minilm["macro_f1"], 4)
)

print("\nMiniLM Hyperparameter Tuning: COMPLETED!")


MINILM HYPERPARAMETER TUNING
C=0.1   | Weight=None     | Accuracy=0.8924 | Macro F1=0.8911
C=0.1   | Weight=balanced | Accuracy=0.9252 | Macro F1=0.9257
C=1.0   | Weight=None     | Accuracy=0.9383 | Macro F1=0.9384
C=1.0   | Weight=balanced | Accuracy=0.9423 | Macro F1=0.9408
C=3.0   | Weight=None     | Accuracy=0.9449 | Macro F1=0.9438
C=3.0   | Weight=balanced | Accuracy=0.9475 | Macro F1=0.9462
C=10.0  | Weight=None     | Accuracy=0.9501 | Macro F1=0.9481
C=10.0  | Weight=balanced | Accuracy=0.9501 | Macro F1=0.9488

MINILM TUNING RESULTS - RANKED


,C,class_weight,accuracy,macro_f1
0,10.0,balanced,0.950131,0.948834
1,10.0,None,0.950131,0.948119
2,3.0,balanced,0.947507,0.946198
3,3.0,None,0.944882,0.943778
4,1.0,balanced,0.942257,0.940806
5,1.0,None,0.938320,0.938368
6,0.1,balanced,0.925197,0.925697
7,0.1,None,0.892388,0.891072



BEST MINILM CLASSIFIER
Best C: 10.0
Best Class Weight: balanced
Best Development Accuracy: 0.9501
Best Development Macro F1: 0.9488

MiniLM Hyperparameter Tuning: COMPLETED!


In [10]:
# ============================================================
# Step 4.7 - Save Pretrained Model & Classifier
# ============================================================

import json
import joblib
from pathlib import Path

# 1. Create model directory
MINILM_DIR = PROJECT_DIR / "models" / "minilm"
MINILM_DIR.mkdir(parents=True, exist_ok=True)

# 2. Define paths
ENCODER_PATH = MINILM_DIR / "sentence_transformer"
CLASSIFIER_PATH = MINILM_DIR / "classifier.joblib"
CONFIG_PATH = MINILM_DIR / "config.json"

# 3. Save pretrained sentence encoder locally
model.save(str(ENCODER_PATH))

print("MiniLM encoder saved!")

# 4. Save trained Logistic Regression classifier
joblib.dump(
    best_minilm_classifier,
    CLASSIFIER_PATH
)

print("Classifier saved!")

# 5. Save configuration
config = {
    "model_name": MODEL_NAME,
    "embedding_dimension": 384,
    "max_sequence_length": model.max_seq_length,
    "normalize_embeddings": True,
    "batch_size": 32,
    "classifier": "LogisticRegression",
    "best_C": best_c_minilm,
    "best_class_weight": best_weight_minilm,
    "random_state": 42,
    "classes": list(best_minilm_classifier.classes_)
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)

print("Configuration saved!")

print("\n" + "=" * 55)
print("SAVED MINILM MODEL FILES")
print("=" * 55)

print("Encoder:", ENCODER_PATH)
print("Classifier:", CLASSIFIER_PATH)
print("Config:", CONFIG_PATH)

print("\nModel saving completed!")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

MiniLM encoder saved!
Classifier saved!
Configuration saved!

SAVED MINILM MODEL FILES
Encoder: C:\Users\faisa\Desktop\NLP Project\models\minilm\sentence_transformer
Classifier: C:\Users\faisa\Desktop\NLP Project\models\minilm\classifier.joblib
Config: C:\Users\faisa\Desktop\NLP Project\models\minilm\config.json

Model saving completed!


In [11]:
# ============================================================
# Step 4.7 - Verify Saved MiniLM Model
# ============================================================

import numpy as np
import json
import joblib

from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, f1_score

print("=" * 55)
print("SAVED MINILM MODEL VERIFICATION")
print("=" * 55)

# 1. Load classifier
loaded_classifier = joblib.load(CLASSIFIER_PATH)

# 2. Load saved configuration
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    loaded_config = json.load(f)

# 3. Verify predictions on existing Dev embeddings
predictions_before = best_minilm_classifier.predict(
    X_dev_embeddings
)

predictions_after = loaded_classifier.predict(
    X_dev_embeddings
)

assert np.array_equal(
    predictions_before,
    predictions_after
)

# 4. Load sentence encoder from local folder
loaded_encoder = SentenceTransformer(
    str(ENCODER_PATH),
    device=DEVICE
)

# 5. Verify end-to-end predictions on sample texts
sample_texts = X_dev.iloc[:5].tolist()

sample_embeddings_original = model.encode(
    sample_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

sample_embeddings_loaded = loaded_encoder.encode(
    sample_texts,
    convert_to_numpy=True,
    normalize_embeddings=loaded_config["normalize_embeddings"]
)

np.testing.assert_allclose(
    sample_embeddings_original,
    sample_embeddings_loaded,
    rtol=1e-4,
    atol=1e-5
)

original_preds = best_minilm_classifier.predict(
    sample_embeddings_original
)

loaded_preds = loaded_classifier.predict(
    sample_embeddings_loaded
)

assert np.array_equal(original_preds, loaded_preds)

# 6. Calculate Development metrics
accuracy = accuracy_score(
    y_dev,
    predictions_after
)

macro_f1 = f1_score(
    y_dev,
    predictions_after,
    average="macro"
)

print("Development Accuracy:", round(accuracy, 4))
print("Development Macro F1:", round(macro_f1, 4))

print("\nEncoder Reloaded: PASS")
print("Classifier Reloaded: PASS")
print("Prediction Consistency: PASS")

print("\nSaved MiniLM Model Integrity: PASSED!")


SAVED MINILM MODEL VERIFICATION


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Development Accuracy: 0.9501
Development Macro F1: 0.9488

Encoder Reloaded: PASS
Classifier Reloaded: PASS
Prediction Consistency: PASS

Saved MiniLM Model Integrity: PASSED!
